In [5]:
import pandas as pd

seg_df = pd.read_csv("../data/processed/churn_risk_segmented.csv")
seg_df.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,churn_probability,risk_segment
0,0,65,94.55,6078.75,True,True,True,True,False,True,...,False,False,False,True,False,True,False,False,0.034962,Low
1,0,26,35.75,1022.50,True,False,False,False,True,False,...,False,False,False,False,False,False,True,False,0.161273,Low
2,0,68,90.20,6297.65,False,True,False,True,False,True,...,False,False,False,True,False,True,False,False,0.038419,Low
3,0,3,84.30,235.05,True,False,False,True,False,False,...,False,True,False,False,False,False,True,False,0.567004,Medium
4,0,49,40.65,2070.75,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,0.151070,Low


In [6]:
prediction_export = seg_df[
    ["churn_probability", "risk_segment", "MonthlyCharges"]
]

prediction_export.to_csv(
    "../data/processed/prediction_export.csv",
    index=False
)

print("prediction_export.csv created")


prediction_export.csv created


In [7]:
segment_cols = [
    col for col in seg_df.columns
    if col.startswith("Contract_")
    or col.startswith("PaymentMethod_")
    or col == "risk_segment"
]

segment_export = seg_df[segment_cols]

segment_export.to_csv(
    "../data/processed/segment_export.csv",
    index=False
)

print("segment_export.csv created")


segment_export.csv created


In [8]:
contract = pd.read_csv("../data/processed/scenario_contract.csv")
discount = pd.read_csv("../data/processed/scenario_discount.csv")
support  = pd.read_csv("../data/processed/scenario_support.csv")

simulation_export = pd.DataFrame({
    "Strategy": ["Contract Upgrade", "Discount", "Tech Support"],
    "Avg_Churn_Probability": [
        contract["new_prob"].mean(),
        discount["new_prob"].mean(),
        support["new_prob"].mean()
    ]
})

simulation_export.to_csv(
    "../data/processed/simulation_export.csv",
    index=False
)

print("simulation_export.csv created")


simulation_export.csv created


In [9]:
revenue_export = seg_df.copy()

revenue_export["annual_at_risk"] = (
    revenue_export["churn_probability"] *
    revenue_export["MonthlyCharges"] *
    12
)

revenue_export = revenue_export[
    ["risk_segment", "MonthlyCharges", "annual_at_risk"]
]

revenue_export.to_csv(
    "../data/processed/revenue_export.csv",
    index=False
)

print("revenue_export.csv created")


revenue_export.csv created


In [10]:
import sqlite3

conn = sqlite3.connect("../data/churn.db")

pd.read_csv("../data/processed/prediction_export.csv") \
  .to_sql("prediction", conn, if_exists="replace", index=False)

pd.read_csv("../data/processed/segment_export.csv") \
  .to_sql("segment", conn, if_exists="replace", index=False)

pd.read_csv("../data/processed/simulation_export.csv") \
  .to_sql("simulation", conn, if_exists="replace", index=False)

pd.read_csv("../data/processed/revenue_export.csv") \
  .to_sql("revenue", conn, if_exists="replace", index=False)

conn.close()

print("SQLite database churn.db created")



SQLite database churn.db created
